<a href="https://colab.research.google.com/github/blakejones-1/Bradford_Jones_Pref_Performance/blob/main/HW2_colab_Jones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Important: please duplicate this file before making modifications:
1. Go to File
2. Save a new copy in Drive/Github


# HW2 | Optimizing Amazon's Product Rankings

Amazon sells products in two ways:  
1. **Amazon-Owned Products** – Items sold directly by Amazon.  
2. **Third-Party Seller Products**  – Items sold by independent businesses on Amazon's platform.  





#Homework Question

`Q5. Can you describe a tradeoff that might exist in how Amazon ranks products?` Please provide the response in the homework document.


## **How the Interactive Model Works:**
1. **Ranking Formula**:  
   - Products are ranked based on **ratings, reviews, and price competitiveness**.  
   - Amazon-owned products can be **boosted** based on revenue weight settings.  

2. **Dynamic Weights**:  
   - You control **how much importance is given** to Amazon vs. third-party revenue.  
   - Adjust **Amazon Revenue Weight (α)** and **Third-Party Revenue Weight (β)** using interactive sliders.  

3. **Live Feedback**:  
   - The **product ranking list updates instantly** based on your selections.  
   - A **bar chart** shows the revenue breakdown for **Amazon vs. third-party sellers**.  


*Run the following two cells*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)


np.random.seed(42)
# Num products ranked
num_products = 20


# Set product features that will be used in the rankings
product_names = [f"Product {i+1}" for i in range(num_products)]
categories = np.random.choice(["Electronics", "Home", "Clothing", "Beauty", "Books"], num_products)
ratings = np.round(np.random.uniform(2.5, 5.0, num_products), 2)
reviews = np.random.randint(10, 500, num_products)
prices = np.round(np.random.uniform(5, 100, num_products), 2)
amazon_owned = np.random.choice([0, 1], num_products, p=[0.6, 0.4])

products_df = pd.DataFrame({
    "Product Name": product_names,
    "Category": categories,
    "Rating": ratings,
    "Reviews": reviews,
    "Price": prices,
    "Amazon Owned (1 = Yes, 0 = No)": amazon_owned
})

In [ ]:
def simulate_purchases(df):
    """Simulate purchase likelihood based on product features."""
    df = df.copy()
    df["Purchase Probability"] = (
        (df["Rating"] / 5) + (np.log2(df["Reviews"] + 1) / 10) - (df["Price"] / 200)
    )

    df["Amazon Purchases"] = df["Purchase Probability"] * df["Amazon Owned (1 = Yes, 0 = No)"] * np.random.uniform(0.8, 1.2)
    df["Third-Party Purchases"] = df["Purchase Probability"] * (1 - df["Amazon Owned (1 = Yes, 0 = No)"]) * np.random.uniform(0.8, 1.2)

    return df

def rank_products(df, amazon_weight=0.5, third_party_weight=0.5):
    """Rank products dynamically based on weighted revenue objectives."""
    df = simulate_purchases(df)

    df["Boosted Score"] = (
        (df["Rating"] * 2) +
        np.log2(df["Reviews"] + 1) -
        (df["Price"] / 50) +
        (amazon_weight * df["Amazon Purchases"]) +
        (third_party_weight * df["Third-Party Purchases"])
    )

    df = df.sort_values(by="Boosted Score", ascending=False).reset_index(drop=True)
    return df

def compute_revenue(df, amazon_weight=0.5, third_party_weight=0.5):
    """Compute weighted total revenue for Amazon vs. Third-Party."""
    amazon_revenue = np.sum(df["Amazon Purchases"] * df["Price"])
    third_party_revenue = np.sum(df["Third-Party Purchases"] * df["Price"])

    weighted_revenue = (amazon_weight * amazon_revenue) + (third_party_weight * third_party_revenue)

    return amazon_revenue, third_party_revenue, weighted_revenue

def update_display(amazon_weight=0.5, third_party_weight=0.5):
    """Dynamically update ranking and revenue results."""
    ranked_df = rank_products(products_df, amazon_weight, third_party_weight)

    amazon_revenue, third_party_revenue, weighted_revenue = compute_revenue(
        ranked_df, amazon_weight, third_party_weight
    )

    print("\n🔹 **Updated Product Rankings** 🔹")
    display(ranked_df[["Product Name", "Category", "Rating", "Amazon Owned (1 = Yes, 0 = No)", "Boosted Score"]])
    print('\n')
    print(f"\n**Amazon Revenue:** ${amazon_revenue:.2f}")
    print(f"**Third-Party Revenue:** ${third_party_revenue:.2f}")
    print(f"**Weighted Total Revenue (Amazon Weight = {amazon_weight}, Third-Party Weight = {third_party_weight}):** ${weighted_revenue:.2f}")
    print('\n')

    plt.figure(figsize=(8, 4))
    labels = ["Amazon-Owned Product Revenue", "Third-Party Product Revenue"]
    values = [amazon_revenue, third_party_revenue]
    plt.bar(labels, values, color=["blue", "green"])
    plt.title("Revenue Breakdown by Source")
    plt.ylabel("Revenue ($)")
    plt.show()

Run the below cell. You can now move the slider bars below to dynamically update the rankings weights.

In [ ]:
interact(update_display,
         amazon_weight=widgets.FloatSlider(min=0, max=2, step=0.1, value=0.5, description="Amazon Revenue Weight"),
         third_party_weight=widgets.FloatSlider(min=0, max=2, step=0.1, value=0.5, description="Third-Party Revenue Weight"))


#Homework Questions

*Please provide the response in the homework document.*

`Q6. What happens as Amazon increases its boost factor?`

`Q7. If Amazon boosts its products too much, do you imagine user engagement will drop? Why?`